# 00 — Instructor Setup (run this ONCE, before the seminar)
### Brain MRI Tumour Segmentation · CS Academy

This notebook turns the 2 GB Kaggle download into a **~70 MB file** that students fetch in one line
with no Kaggle account.

Do this at least a week before the seminar. Students never open this notebook.

**Steps:**
1. Download the raw dataset (needs your Kaggle token — yours, once, not theirs)
2. Convert to `lgg_128.npz`
3. Sanity-check it
4. Host it somewhere public and paste the URL into `seminar.py`

In [ ]:
#@title 1. Get seminar.py
REPO_RAW = "https://raw.githubusercontent.com/OTMAN-REPO/brain-mri-seminar/main"  #@param {type:"string"}
import os, urllib.request
if not os.path.exists("seminar.py"):
    urllib.request.urlretrieve(f"{REPO_RAW}/seminar.py", "seminar.py")
from seminar import *
import numpy as np
print("ok")

## 2. Download the raw dataset

Get your token from **kaggle.com → your profile → Settings → API → Create New Token**.
That downloads `kaggle.json`. Upload it here with the folder icon on the left.

In [ ]:
!pip -q install kagglehub
import os, glob, shutil
if os.path.exists("kaggle.json"):
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    shutil.copy("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

import kagglehub
src = kagglehub.dataset_download("mateuszbuda/lgg-mri-segmentation")
print("downloaded to:", src)

root = None
for d, subdirs, _ in os.walk(src):
    if any(s.startswith("TCGA_") for s in subdirs):
        root = d; break
print("patient folders in:", root)
print("patients found:", len([s for s in os.listdir(root) if s.startswith("TCGA_")]))

## 3. Convert

128×128 is the recommended size: it trains in seconds per epoch on a free T4 and the tumours are
still clearly resolved. Build a 256 version too if you want an "does resolution matter?" experiment.

In [ ]:
build_npz(root, "lgg_128.npz", size=128)
imgs, msks, pids, sidx = load_npz("lgg_128.npz")

print(f"\nslices           : {len(imgs)}")
print(f"patients         : {len(np.unique(pids))}")
print(f"image shape      : {imgs.shape[1:]}")
print(f"slices w/ tumour : {(msks.reshape(len(msks),-1).sum(1) > 0).mean():.1%}")
print(f"tumour pixels    : {msks.mean():.3%}")
print(f"file size        : {os.path.getsize('lgg_128.npz')/1e6:.1f} MB")

print("\nExpected, roughly: ~3900 slices, 110 patients, ~1/3 slices with tumour, ~1% tumour pixels.")
print("If those are wildly off, something went wrong in the conversion.")

In [ ]:
#@title Also save the genomic/clinical table for capstone track C
import pandas as pd, glob
csvs = glob.glob(os.path.join(src, "**", "data.csv"), recursive=True)
if csvs:
    pd.read_csv(csvs[0]).to_csv("lgg_meta.csv", index=False)
    print("saved lgg_meta.csv", pd.read_csv("lgg_meta.csv").shape)
else:
    print("data.csv not found — track C will fall back to demo mode")

In [ ]:
#@title 4. Visual sanity check — DO look at this
import matplotlib.pyplot as plt
areas = msks.reshape(len(msks),-1).sum(1)
picks = np.argsort(areas)[-8:]
fig, ax = plt.subplots(1, 8, figsize=(18, 2.6))
for k, j in enumerate(picks):
    ax[k].imshow(imgs[j][...,1], cmap="gray")
    ax[k].contour(msks[j], levels=[.5], colors="lime", linewidths=1.2)
    ax[k].set_title(pids[j].split("_")[2], fontsize=8); ax[k].axis("off")
plt.suptitle("masks must line up with the bright lesions. If they are offset, the resize broke.")
plt.tight_layout(); plt.show()

## 5. Host it

Pick whichever is easiest for you. Students need a URL that downloads the file directly with no login.

**Hugging Face (recommended — free, fast, no auth, no quota problems):**
```python
from huggingface_hub import HfApi
api = HfApi(token="hf_...")
api.create_repo("YOUR_NAME/lgg-seminar", repo_type="dataset")
api.upload_file(path_or_fileobj="lgg_128.npz", path_in_repo="lgg_128.npz",
                repo_id="YOUR_NAME/lgg-seminar", repo_type="dataset")
```
URL becomes:
`https://huggingface.co/datasets/YOUR_NAME/lgg-seminar/resolve/main/lgg_128.npz`

**GitHub Release** — a 70 MB asset is fine (100 MB limit). Don't commit it to the repo itself.

**Google Drive** — works, but the confirm-download interstitial breaks `urlretrieve` for large files.
Only use it as a last resort.

### Then, the important bit

Open `seminar.py`, set `DATA_URL` and `META_URL` to those links, and commit it. Every student
notebook picks them up automatically and the setup cell becomes a no-op for them.

In [ ]:
#@title 6. Final check — pretend you are a student
# Delete the local file and make sure the URL path works from scratch.
import os
TEST_URL = ""  #@param {type:"string"}
if TEST_URL:
    if os.path.exists("lgg_128.npz"): os.rename("lgg_128.npz", "_backup.npz")
    i, m, p, s = get_data(url=TEST_URL, allow_demo=False)
    print(f"\nSUCCESS: {len(i)} slices, {len(np.unique(p))} patients")
else:
    print("Paste your hosted URL above and re-run.")

---
## Before you teach: do a full dry run

Run notebooks 01→05 end to end on the real data and **write down the numbers you get.** You need
these, because students will ask "is my number right?" and the answer depends on your setup.

Specifically record:

| | what to record |
|---|---|
| Session 2 | best Dice from thresholding |
| Session 3 | U-Net Dice on the **slice** split |
| Session 4 | U-Net Dice on the **patient** split, and the gap between them |
| Session 4 | how long 12 epochs takes on a T4 |

**The Session 4 gap is the one to check most carefully.** The whole session is built around it. It
should be a clear drop, and it should be larger than the seed-to-seed variation. If it isn't for
your setup, adjust the epoch count or val fraction until the effect is visible and stable —
or be ready to teach it as "sometimes leakage is subtle, which is what makes it dangerous."

For reference, the original paper reports **83.6% mean Dice** with 22-fold cross-validation
(Buda et al. 2019), and puts human inter-reader agreement at **~84%, sd 2%**. Students training for
12 epochs on a subset will land below that — which is fine and worth saying out loud.